# Predictive Healthcare MEDS - Exploratory Data Analysis

This notebook performs exploratory data analysis on the healthcare dataset,
examining patient demographics, vital signs distributions, readmission patterns,
and feature correlations.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_pipeline.data_loader import DataLoader
from src.utils.config_loader import load_config

# Plot settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Libraries loaded successfully')

## 1. Load Data

In [ ]:
# Load configuration and data
data_config = load_config('configs/data_config.yaml')
loader = DataLoader(data_config)

# Try loading from file, or generate dummy data
df = loader.load()

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Basic Statistics

In [ ]:
# Overview of numeric columns
df.describe()

In [ ]:
# Data types and missing values
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

## 3. Target Variable Analysis

In [ ]:
# Readmission distribution
target = 'readmission_30day'
if target in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count plot
    counts = df[target].value_counts()
    axes[0].bar(['No Readmission', 'Readmitted'], counts.values, color=['#2ecc71', '#e74c3c'])
    axes[0].set_title('30-Day Readmission Distribution')
    axes[0].set_ylabel('Count')
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')
    
    # Percentage
    pct = df[target].mean() * 100
    axes[1].pie([100-pct, pct], labels=['No Readmission', 'Readmitted'],
                colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
    axes[1].set_title(f'Readmission Rate: {pct:.1f}%')
    
    plt.tight_layout()
    plt.show()

## 4. Demographic Analysis

In [ ]:
# Age distribution by readmission status
if 'age' in df.columns and target in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for label, group in df.groupby(target):
        name = 'Readmitted' if label == 1 else 'Not Readmitted'
        axes[0].hist(group['age'], bins=30, alpha=0.6, label=name)
    axes[0].set_title('Age Distribution by Readmission')
    axes[0].set_xlabel('Age')
    axes[0].legend()
    
    # Readmission rate by age group
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 35, 50, 65, 80, 120],
                              labels=['0-18', '19-35', '36-50', '51-65', '66-80', '80+'])
    rate_by_age = df.groupby('age_group')[target].mean()
    axes[1].bar(rate_by_age.index.astype(str), rate_by_age.values, color='#3498db')
    axes[1].set_title('Readmission Rate by Age Group')
    axes[1].set_ylabel('Readmission Rate')
    axes[1].set_xlabel('Age Group')
    
    plt.tight_layout()
    plt.show()

## 5. Vital Signs Distribution

In [ ]:
# Distribution of key vital signs
vitals = ['heart_rate', 'blood_pressure_systolic', 'oxygen_saturation', 'temperature']
available_vitals = [v for v in vitals if v in df.columns]

if available_vitals:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, vital in enumerate(available_vitals[:4]):
        for label, group in df.groupby(target):
            name = 'Readmitted' if label == 1 else 'Not Readmitted'
            axes[i].hist(group[vital].dropna(), bins=30, alpha=0.6, label=name)
        axes[i].set_title(f'{vital} Distribution')
        axes[i].set_xlabel(vital)
        axes[i].legend()
    
    plt.tight_layout()
    plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr()
    
    fig, ax = plt.subplots(figsize=(14, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', center=0,
                linewidths=0.5, ax=ax)
    ax.set_title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 7. Comorbidity Analysis

In [ ]:
# Readmission rate by comorbidity
comorbidities = ['has_diabetes', 'has_hypertension', 'has_chd', 'has_copd', 'has_renal_disease']
available_comorbid = [c for c in comorbidities if c in df.columns]

if available_comorbid and target in df.columns:
    rates = {}
    for col in available_comorbid:
        rate_with = df[df[col] == 1][target].mean()
        rate_without = df[df[col] == 0][target].mean()
        rates[col] = {'With': rate_with, 'Without': rate_without}
    
    rates_df = pd.DataFrame(rates).T
    
    fig, ax = plt.subplots(figsize=(10, 6))
    rates_df.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
    ax.set_title('Readmission Rate by Comorbidity Status')
    ax.set_ylabel('Readmission Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.legend(title='Comorbidity')
    
    plt.tight_layout()
    plt.show()

## 8. Key Takeaways

Document your findings from the EDA here:
- Dataset size and characteristics
- Class balance of the target variable
- Key features correlated with readmission
- Missing value patterns
- Potential feature engineering opportunities